In [1]:
# Javier Tovar and Victor Z
# UW-Madison, 2024
# flare sizing

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Specify the path of this file (This may differ for each user)
path = '/content/drive/MyDrive/Colab Notebooks/'

In [11]:
# -------------------------------
# Load input mass flow data (lb/h)
# -------------------------------
filename=path+'flow_flare.dat'
M = np.loadtxt(filename)
S = len(M)

In [18]:
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd

# -------------------------------
# Design specifications
# -------------------------------

Kmax = 3000     # Maximum allowable radiation BTU/(h*ft^2)
Mamax = 0.7     # Maximum allowable Mach number (-)

# Design variables
diam = 1.0      # Diameter (ft)
height = 20.0   # Height of flare stack (ft)

# -------------------------------
# Evaluate flare design at different flows
# -------------------------------
cost = np.zeros(S)
Ma = np.zeros(S)
K = np.zeros(S)

for i in range(S):
    cost[i], Ma[i], K[i], _ = flare_model(diam, height, M[i])

# -------------------------------
# Flare model function
# -------------------------------
def flare_model(diam, height, M):
    """
    Compute flare performance metrics and cost.

    Inputs:
        diam   : Flare diameter (ft)
        height : Flare stack height (ft)
        M      : Mass flow (lb/h)

    Outputs:
        cost   : Cost (USD)
        Ma     : Mach number (-)
        K      : Radiation (BTU/(h*ft^2))
        U      : Exit velocity (ft/s)
    """
    # Model parameters
    mw = 46.1         # molecular weight of reactant (lb/lbmol)
    t = 760.0         # temperature (°R)
    z = 1.0           # compressibility factor (-)
    hc = 21500.0      # heat of combustion (btu/lb)
    p = 14.7          # absolute pressure (psi)
    w = 29.3          # wind velocity (ft/s)
    f = 0.3           # fraction of heat radiated (-)
    rad = 150.0       # distance to flare stack (ft)

    hv = mw * hc / 379.5  # heating value (btu/scf)

    # Model equations
    H = M * hc
    L = 10 ** (0.4507 * math.log10(H) - 1.9885)
    Q = (M / 3600.0) * (379.1 / mw) * (t / 520.0)

    Ma = 1.702e-5 * (M / (p * diam**2)) * math.sqrt(z * t / mw)
    U = 4 * Q / (math.pi * diam**2)

    DX = math.exp(math.log(L * 0.9838) + 0.0754 * (math.log(w / U)))
    DY = math.exp(math.log(L * 0.0985) - 0.705 * (math.log(w / U)))

    Hr = height + 0.5 * DY
    Rr = rad - 0.5 * DX
    D = math.sqrt(Rr**2 + Hr**2)

    K = f * H / (D**2 * 4 * math.pi)
    cost = (94.3 + 11.05 * (diam * 12.0) + 0.906 * height)**2

    return cost, Ma, K, U

In [20]:
pd.DataFrame(M).head(5)


,0
0,17120.0830
1,62581.5250
2,1044.7088
3,23683.0220
4,13754.2840


In [21]:
pd.DataFrame(Ma).head(5)

,0
0,0.080483
1,0.294201
2,0.004911
3,0.111336
4,0.064660


In [22]:
pd.DataFrame(K).head(5)

,0
0,621.770082
1,2788.655398
2,27.554203
3,913.481310
4,480.797600
